In [0]:
# Configuration

CATALOG = "adb_retailedge_dev"
SCHEMA  = "healthcare_cms"

print("Gold tables available:")
for t in ["gold_provider_performance", "gold_drug_spend_analysis",
        "gold_diagnosis_trends", "gold_fraud_indicators"]:
    count = spark.table(f"{CATALOG}.{SCHEMA}.{t}").count()
    print(f"  {t}: {count:,} rows")

Gold tables available:
  gold_provider_performance: 3,435 rows
  gold_drug_spend_analysis: 14,536 rows
  gold_diagnosis_trends: 25,804 rows
  gold_fraud_indicators: 9,184 rows


## Q1: Which specialties cost Medicare the most? (Top 10 nationally)
- **What this answers:** Which medical specialties cost Medicare the most on average per service — nationally across all states

In [0]:

from pyspark.sql.functions import col, round as spark_round, sum as spark_sum, avg as spark_avg

print("=" * 60)
print("Q1: TOP 10 MOST EXPENSIVE SPECIALTIES (National Average)")
print("=" * 60)

display(
    spark.table(f"{CATALOG}.{SCHEMA}.gold_provider_performance")
    .groupBy("provider_type")
    .agg(
        spark_sum("total_providers").alias("total_providers"),
        spark_round(spark_avg("avg_medicare_payment"), 2).alias("avg_medicare_payment"),
        spark_round(spark_avg("avg_submitted_charge"), 2).alias("avg_submitted_charge")
    )
    .orderBy(col("avg_medicare_payment").desc())
    .limit(10)
  )

Q1: TOP 10 MOST EXPENSIVE SPECIALTIES (National Average)


provider_type,total_providers,avg_medicare_payment,avg_submitted_charge
Ambulatory Surgical Center,334,1104.18,6386.93
Thoracic Surgery,141,404.34,1851.75
Ambulance Service Provider,557,346.85,1803.57
Cardiac Surgery,57,317.3,1675.57
Radiation Therapy Center,5,301.56,3655.46
Micrographic Dermatologic Surgery,27,215.9,738.95
Neurosurgery,326,214.44,1452.23
Opioid Treatment Program,56,166.29,217.14
Independent Diagnostic Testing Facility (IDTF),114,163.89,1271.01
Vascular Surgery,238,153.54,646.95


## Q2: Top 10 drugs by spending + fastest growing prices
**What this answers:**
  - Q2A — Where is Medicare spending the most money on drugs?
  - Q2B — Which drug prices are rising the fastest (>10% per year)?

In [0]:
from pyspark.sql.functions import col

print("=" * 60)
print("Q2A: TOP 10 DRUGS BY TOTAL MEDICARE SPENDING (2024)")
print("=" * 60)

display(
    spark.table(f"{CATALOG}.{SCHEMA}.gold_drug_spend_analysis")
    .select("brand_name", "generic_name", "total_spending_2024",
            "total_claims_2024", "avg_spend_per_claim_2024", "price_trend_category")
    .orderBy(col("total_spending_2024").desc())
    .limit(10)
)

print("=" * 60)
print("Q2B: PRICE TREND CATEGORY BREAKDOWN")
print("=" * 60)

display(
    spark.table(f"{CATALOG}.{SCHEMA}.gold_drug_spend_analysis")
    .groupBy("price_trend_category")
    .count()
    .orderBy("count", ascending=False)
)

print("=" * 60)
print("Q2C: TOP 10 FASTEST GROWING DRUG PRICES (BY CAGR)")
print("=" * 60)

display(
    spark.table(f"{CATALOG}.{SCHEMA}.gold_drug_spend_analysis")
    .filter(col("cagr_2020_2024").isNotNull())
    .select("brand_name", "generic_name", "cagr_2020_2024",
            "total_spending_2024", "total_spending_2022", "price_trend_category")
    .orderBy(col("cagr_2020_2024").desc())
    .limit(10)
)

Q2A: TOP 10 DRUGS BY TOTAL MEDICARE SPENDING (2024)


brand_name,generic_name,total_spending_2024,total_claims_2024,avg_spend_per_claim_2024,price_trend_category
Eliquis,Apixaban,2.0774929225E10,24061332,863.42,Moderate Growth
Eliquis,Apixaban,2.0774929225E10,24061332,863.42,Moderate Growth
Ozempic,Semaglutide,1.2970296347E10,10417182,1245.09,Stable
Ozempic,Semaglutide,1.2970296347E10,10417182,1245.09,Stable
Jardiance,Empagliflozin,1.1435997421E10,11368280,1005.96,Stable
Jardiance,Empagliflozin,1.1435997421E10,11368280,1005.96,Stable
Mounjaro,Tirzepatide,6.3369426407E9,5105397,1241.22,Stable
Mounjaro,Tirzepatide,6.3369426407E9,5105397,1241.22,Stable
Xarelto,Rivaroxaban,6.2322318397E9,6660246,935.74,Stable
Xarelto,Rivaroxaban,6.2322318397E9,6660246,935.74,Stable


Q2B: PRICE TREND CATEGORY BREAKDOWN


price_trend_category,count
Stable,11698
Unknown,1160
Moderate Growth,870
High Growth,808


Q2C: TOP 10 FASTEST GROWING DRUG PRICES (BY CAGR)


brand_name,generic_name,cagr_2020_2024,total_spending_2024,total_spending_2022,price_trend_category
Pantoprazole Sodium*,Pantoprazole Sodium,12.5,15524.08,null,High Growth
Indomethacin*,Indomethacin,10.9177,2018.81,1839.36,High Growth
Jynneos,Smallpox And Mpox Live Vacc/PF,8.7159,337011.89,null,High Growth
Jynneos,Smallpox And Mpox Live Vacc/PF,8.7159,337011.89,null,High Growth
Metformin HCl,Metformin HCl,7.9985,441633.29,1061.85,High Growth
Lagevrio (Eua),Molnupiravir,6.7226,1.2071377357E8,3486845.94,High Growth
Lagevrio (Eua),Molnupiravir,6.7226,1.2071377357E8,3486845.94,High Growth
Indomethacin*,Indomethacin,4.6893,518342.99,null,High Growth
Ganciclovir Sodium,Ganciclovir Sodium,3.6169,18580.18,32800.5,High Growth
Dextroamphetamine Sulfate,Dextroamphetamine Sulfate,3.2963,344815.49,null,High Growth


In [0]:
  from pyspark.sql.functions import col

  print("=" * 60)
  print("Q2A: TOP 10 DRUGS BY TOTAL MEDICARE SPENDING (2024)")
  print("=" * 60)

  display(
      spark.table(f"{CATALOG}.{SCHEMA}.gold_drug_spend_analysis")
      .select("brand_name", "generic_name", "total_spending_2024",
              "total_claims_2024", "avg_spend_per_claim_2024", "price_trend_category")
      .dropDuplicates(["brand_name", "generic_name"])
      .orderBy(col("total_spending_2024").desc())
      .limit(10)
  )

  print("=" * 60)
  print("Q2B: TOP 10 FASTEST GROWING DRUG PRICES (CAGR > 10% per year)")
  print("=" * 60)

  display(
      spark.table(f"{CATALOG}.{SCHEMA}.gold_drug_spend_analysis")
      .filter(col("cagr_2020_2024").isNotNull())
      .dropDuplicates(["brand_name", "generic_name"])
      .select("brand_name", "generic_name", "cagr_2020_2024",
              "total_spending_2024", "total_spending_2022")
      .orderBy(col("cagr_2020_2024").desc())
      .limit(10)
  )


Q2A: TOP 10 DRUGS BY TOTAL MEDICARE SPENDING (2024)


brand_name,generic_name,total_spending_2024,total_claims_2024,avg_spend_per_claim_2024,price_trend_category
Eliquis,Apixaban,2.0774929225E10,24061332,863.42,Moderate Growth
Ozempic,Semaglutide,1.2970296347E10,10417182,1245.09,Stable
Jardiance,Empagliflozin,1.1435997421E10,11368280,1005.96,Stable
Mounjaro,Tirzepatide,6.3369426407E9,5105397,1241.22,Stable
Xarelto,Rivaroxaban,6.2322318397E9,6660246,935.74,Stable
Trulicity,Dulaglutide,5.4541347251E9,4305141,1266.89,Stable
Trelegy Ellipta,Fluticasone/Umeclidin/Vilanter,5.2944017768E9,6250872,846.99,Stable
Farxiga,Dapagliflozin Propanediol,5.2905489634E9,5634650,938.93,Stable
Humira(Cf) Pen,Adalimumab,4.3286988642E9,490303,8828.62,Moderate Growth
Revlimid,Lenalidomide,4.1795114863E9,239843,17426.03,Stable


Q2B: TOP 10 FASTEST GROWING DRUG PRICES (CAGR > 10% per year)


brand_name,generic_name,cagr_2020_2024,total_spending_2024,total_spending_2022
Jynneos,Smallpox And Mpox Live Vacc/PF,8.7159,337011.89,null
Lagevrio (Eua),Molnupiravir,6.7226,1.2071377357E8,3486845.94
Amphotericin B Liposome,Amphotericin B Liposome,1.5373,1862373.94,371248.53
Recarbrio,Imipenem/Cilastatin/Relebactam,1.2568,1226579.25,1125796.34
Avodart,Dutasteride,1.0567,6811784.16,null
Prednisolone,Prednisolone,0.9147,1919216.27,1308721.91
Trudhesa,Dihydroergotamine Mesylate,0.8439,5646780.53,1534221.85
Unasyn,Ampicillin Sod/Sulbactam Sod,0.8094,84426.83,90943.34
Oxaliplatin,Oxaliplatin,0.7333,943032.07,781540.37
Lidocaine-Hydrocortisone*,Lidocaine/Hydrocortisone Ac,0.7113,1577.09,null


## Q3: Disease trends by state.

In [0]:
from pyspark.sql.functions import col

print("=" * 60)
print("Q3A: TOP 10 MOST COMMON DIAGNOSES NATIONALLY")
print("=" * 60)

display(
    spark.table(f"{CATALOG}.{SCHEMA}.gold_diagnosis_trends")
    .groupBy("drg_code", "drg_description")
    .sum("total_discharges")
    .withColumnRenamed("sum(total_discharges)", "national_discharges")
    .orderBy(col("national_discharges").desc())
    .limit(10)
)

print("=" * 60)
print("Q3B: TOP 5 MOST EXPENSIVE DIAGNOSES (AVG MEDICARE PAYMENT)")
print("=" * 60)

display(
    spark.table(f"{CATALOG}.{SCHEMA}.gold_diagnosis_trends")
    .groupBy("drg_code", "drg_description")
    .avg("avg_medicare_payment")
    .withColumnRenamed("avg(avg_medicare_payment)", "avg_medicare_payment")
    .orderBy(col("avg_medicare_payment").desc())
    .limit(5)
)

Q3A: TOP 10 MOST COMMON DIAGNOSES NATIONALLY


drg_code,drg_description,national_discharges
871,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITH MCC,578073
291,HEART FAILURE AND SHOCK WITH MCC,306135
177,RESPIRATORY INFECTIONS AND INFLAMMATIONS WITH MCC,144162
193,SIMPLE PNEUMONIA AND PLEURISY WITH MCC,139034
872,SEPTICEMIA OR SEVERE SEPSIS WITHOUT MV >96 HOURS WITHOUT MCC,107314
690,KIDNEY AND URINARY TRACT INFECTIONS WITHOUT MCC,95596
189,PULMONARY EDEMA AND RESPIRATORY FAILURE,93765
392,"ESOPHAGITIS, GASTROENTERITIS AND MISCELLANEOUS DIGESTIVE DISORDERS WITHOUT MCC",90637
280,"ACUTE MYOCARDIAL INFARCTION, DISCHARGED ALIVE WITH MCC",87661
689,KIDNEY AND URINARY TRACT INFECTIONS WITH MCC,84058


Q3B: TOP 5 MOST EXPENSIVE DIAGNOSES (AVG MEDICARE PAYMENT)


drg_code,drg_description,avg_medicare_payment
018,CHIMERIC ANTIGEN RECEPTOR (CAR) T-CELL AND OTHER IMMUNOTHERAPIES,441724.4429032258
927,EXTENSIVE BURNS OR FULL THICKNESS BURNS WITH MV >96 HOURS WITH SKIN GRAFT,345341.33
001,HEART TRANSPLANT OR IMPLANT OF HEART ASSIST SYSTEM WITH MCC,261485.66027777776
003,"ECMO OR TRACHEOSTOMY WITH MV >96 HOURS OR PRINCIPAL DIAGNOSIS EXCEPT FACE, MOUTH AND NEC",175946.10326530613
007,LUNG TRANSPLANT,136472.1882352941


In [0]:
from pyspark.sql.functions import col

display(
      spark.table(f"{CATALOG}.{SCHEMA}.gold_fraud_indicators")
      .filter(col("fraud_risk_flag") == "High Risk")
      .select("provider_npi", "provider_name", "provider_first_name",
              "state", "provider_type",
              "avg_medicare_payment", "specialty_avg_payment",
              "deviation_from_avg", "stddev_multiplier")
      .orderBy(col("stddev_multiplier").desc())
      .limit(10)
  )

provider_npi,provider_name,provider_first_name,state,provider_type,avg_medicare_payment,specialty_avg_payment,deviation_from_avg,stddev_multiplier
1043087182,Hyde,Tiffany,IL,Nurse Practitioner,1203.89,64.78,1139.11,18.81
1053637355,Garrison,Lyddia,IL,Physical Therapist in Private Practice,337.39,30.95,306.44,14.78
1043682065,Williams,Nikia,FL,Nurse Practitioner,547.62,66.17,481.45,14.61
1013501105,Chapler,Chana,NJ,Nurse Practitioner,593.47,70.26,523.21,13.7
1033657937,Smith-Litchfield,Nadia,NY,Nurse Practitioner,387.88,60.52,327.36,12.28
1053634493,Bishop,Yelena,CA,Physician Assistant,577.35,72.04,505.31,11.8
1053859421,Roark,Sarah,CO,Nurse Practitioner,538.46,64.52,473.94,11.39
1043518186,Prasad,Keerthi,TX,Diagnostic Radiology,866.45,44.95,821.5,11.32
1003985177,Niku,Soheil,CA,Diagnostic Radiology,876.44,60.47,815.97,11.28
1013172907,Lakhanpal,Gaurav,MD,Internal Medicine,891.3,85.91,805.39,11.04


## Q4B: FRAUD RISK BY STATE — WHICH STATES HAVE MOST HIGH RISK PROVIDERS?

In [0]:
print("=" * 60)
print("Q4B: FRAUD RISK BY STATE — WHICH STATES HAVE MOST HIGH RISK PROVIDERS?")
print("=" * 60)

display(
    spark.table(f"{CATALOG}.{SCHEMA}.gold_fraud_indicators")
      .filter(col("fraud_risk_flag") == "High Risk")
      .groupBy("state")
      .count()
      .withColumnRenamed("count", "high_risk_providers")
      .orderBy(col("high_risk_providers").desc())
      .limit(10)
  )


Q4B: FRAUD RISK BY STATE — WHICH STATES HAVE MOST HIGH RISK PROVIDERS?


state,high_risk_providers
CA,180
TX,170
NY,159
FL,139
PA,122
OH,96
NC,79
MI,72
MA,67
IL,64


## Q4C: FRAUD RISK BY SPECIALTY — WHICH SPECIALTIES HAVE MOST HIGH RISK?

In [0]:

  print("=" * 60)
  print("Q4C: FRAUD RISK BY SPECIALTY — WHICH SPECIALTIES HAVE MOST HIGH RISK?")
  print("=" * 60)

  display(
      spark.table(f"{CATALOG}.{SCHEMA}.gold_fraud_indicators")
      .filter(col("fraud_risk_flag") == "High Risk")
      .groupBy("provider_type")
      .count()
      .withColumnRenamed("count", "high_risk_providers")
      .orderBy(col("high_risk_providers").desc())
      .limit(10)
  )

Q4C: FRAUD RISK BY SPECIALTY — WHICH SPECIALTIES HAVE MOST HIGH RISK?


provider_type,high_risk_providers
Nurse Practitioner,335
Physician Assistant,206
Internal Medicine,136
Certified Registered Nurse Anesthetist (CRNA),119
Family Practice,110
Physical Therapist in Private Practice,103
Anesthesiology,91
Optometry,70
Diagnostic Radiology,69
Orthopedic Surgery,50
